# VRC-3PO — Session-Scale Temporal Context

Every model in the article operates inside one 30-second window and carries no state
from one window to the next. Nothing it computes can depend on **where in the exposure**
a window sits. This notebook tests whether that costs anything.

**The design decision that makes this cheap.** The session is *not* the prediction unit.
The window stays the prediction unit — the same 768 single-task held-out windows, the same
pooled AUC — and only what the model can *see* changes. A causal GRU runs over the sequence
of window embeddings within a session, so the prediction for window *t* may depend on
windows *1…t* but never on the future. Sessions are a median of 25 windows long, so this is
a 25-step recurrence, not a 1,000-step one.

That keeps the result directly comparable to the frozen **AUC 0.757** and to the
20-composition spread (mean 0.631, range 0.468–0.748).

## Conditions

| Key | What it is | What a change tells you |
|---|---|---|
| `window` | Frozen architecture, retrained here | Sanity anchor; should land near 0.757 |
| `window_clock` | Same + normalized position-in-session feature | How much of any gain is just a clock |
| `session` | Causal GRU over window embeddings | The actual hypothesis |
| `session_shuffled` | Same, window order shuffled within session | **The decisive control.** If this holds up, the gain came from pooling, not temporal order |
| `session_prefix_k` | Context truncated to *k* windows | Which temporal scale carries the signal |

A gain in `session` that survives `session_shuffled` and is not explained by `window_clock`
is evidence for accumulation. A gain that dies under either is still a finding, and a more
interesting one than silence.

## Honesty constraints built in

- **Alignment guard.** The notebook rebuilds the window index independently and refuses to
  proceed unless it reproduces the frozen ordering (pooled AUC 0.756948).
- **Participant-held-out only.** The purged-blocked protocol splits *within* a session, so a
  causal session model would consume the training block as test-time context. Not run here.
- **Single-task subset.** Train, validation and test are restricted to Simulations and
  Terrain, matching `PROTOCOL_VERSION = 2` and the reference endpoint.
- Intervals resample **whole participants**, never windows.

## Cost
40 fits (8 conditions x 5 seeds). Roughly 2–4 hours on a T4.

## What you need in Drive
- `vrc3po_master_dataset_fixed.csv`
- `results/corrected/split_manifest.csv`
- `results/corrected/cnn_ensemble_preds_OG.npy` (for the alignment guard)

## Cell 1 — Setup and configuration

In [ ]:
# Cell 1: setup. Edit PROJECT_ROOT to point at your VRC-3PO folder in Drive.
import os, sys, json, time, warnings
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = Path("/content/drive/MyDrive/VRC-3PO")
except Exception:
    PROJECT_ROOT = Path(".")          # local fallback

DATASET_CSV    = PROJECT_ROOT / "vrc3po_master_dataset_fixed.csv"
SPLIT_MANIFEST = PROJECT_ROOT / "results/corrected/split_manifest.csv"
FROZEN_PREDS   = PROJECT_ROOT / "results/corrected/cnn_ensemble_preds_OG.npy"
OUTPUT_DIR     = PROJECT_ROOT / "results/session_scale"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Frozen protocol constants -- do not change these without bumping PROTOCOL_VERSION.
WINDOW_LENGTH, WINDOW_STRIDE = 30, 15
SINGLE_TASK_SOURCES = ("simulations", "terrain")
ENSEMBLE_SEEDS      = (42, 123, 456, 789, 2024)
FROZEN_POOLED_AUC   = 0.756947866036354
BOOTSTRAP_REPLICATES = 5000
PROTOCOL_VERSION    = 3   # v3: session-scale context, single-task, participant-held-out

FEATURE_COLUMNS = [
    "pupil_diam_L", "pupil_diam_R", "eye_open_L", "eye_open_R",
    "gaze_dir_world_X", "gaze_dir_world_Y", "gaze_dir_world_Z",
    "gaze_origin_world_X", "gaze_origin_world_Y", "gaze_origin_world_Z",
    "head_quat_X", "head_quat_Y", "head_quat_Z", "head_quat_W",
]

for path in (DATASET_CSV, SPLIT_MANIFEST):
    if not path.exists():
        raise SystemExit(f"missing required input: {path}")
if not FROZEN_PREDS.exists():
    warnings.warn(f"{FROZEN_PREDS} not found -- the alignment guard will be skipped. "
                  "Results cannot be claimed comparable to the frozen endpoint.")

import numpy as np, pandas as pd
print("numpy", np.__version__, "| pandas", pd.__version__)
try:
    import tensorflow as tf
    print("tensorflow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))
except ImportError:
    raise SystemExit("TensorFlow not available -- switch the Colab runtime to GPU.")
print("project root:", PROJECT_ROOT)

## Cell 2 — Build the session-grouped window index, and verify it against the frozen ordering

In [ ]:
# Cell 2: rebuild windows grouped by session, then prove the index matches the frozen one.
import numpy as np, pandas as pd

dataset = pd.read_csv(DATASET_CSV)
split   = pd.read_csv(SPLIT_MANIFEST)

def build_session_windows(df):
    """30 s windows at 15 s stride, kept grouped by session and in temporal order.

    Identical windowing to the frozen pipeline; the only difference is that we
    retain the session grouping and each window's position within it.
    """
    sessions = []
    for (participant, condition), session in df.groupby(
            ["global_participant_id", "condition"], sort=True):
        session = session.sort_values("elapsed_s").reset_index(drop=True)
        feats = session[FEATURE_COLUMNS].to_numpy(dtype=np.float32)
        fms   = session["fms"].to_numpy(dtype=np.float32)
        elapsed = session["elapsed_s"].to_numpy(dtype=np.float64)
        starts = range(0, len(session) - WINDOW_LENGTH + 1, WINDOW_STRIDE)
        w_feats, w_labels, w_time = [], [], []
        for s in starts:
            w_feats.append(feats[s:s + WINDOW_LENGTH])
            w_labels.append(float(fms[s:s + WINDOW_LENGTH].mean()))
            w_time.append(elapsed[s])
        if not w_feats:
            continue
        span = max(w_time[-1] - w_time[0], 1.0)
        sessions.append({
            "participant": str(participant),
            "condition": str(condition),
            "source": str(session["source_dataset"].iloc[0]),
            "features": np.stack(w_feats),                      # (n_windows, 30, 14)
            "labels":   np.asarray(w_labels, dtype=np.float32),  # (n_windows,)
            "position": np.asarray([(t - w_time[0]) / span for t in w_time],
                                   dtype=np.float32),            # 0..1 within session
        })
    return sessions

SESSIONS = build_session_windows(dataset)
split_of = split.set_index("participant_id")["split"].to_dict()
for s in SESSIONS:
    s["split"] = split_of.get(s["participant"])

n_windows = sum(len(s["labels"]) for s in SESSIONS)
lengths   = np.array([len(s["labels"]) for s in SESSIONS])
print(f"sessions {len(SESSIONS)} | windows {n_windows}")
print(f"windows per session: median {np.median(lengths):.0f}  min {lengths.min()}  max {lengths.max()}")
assert n_windows == 6442, f"expected 6442 windows, rebuilt {n_windows}"

# ---- alignment guard -------------------------------------------------------
# Flatten the single-task test windows in the same order the frozen predictions
# use (participant, condition, time) and check the pooled AUC reproduces exactly.
def rank_auc(y, score):
    y = np.asarray(y); r = pd.Series(np.asarray(score)).rank().to_numpy()
    n1 = y.sum(); n0 = len(y) - n1
    return float("nan") if n1 == 0 or n0 == 0 else float(
        (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

test_single = sorted(
    [s for s in SESSIONS if s["split"] == "test" and s["source"] in SINGLE_TASK_SOURCES],
    key=lambda s: (s["participant"], s["condition"]))
y_test_flat = np.concatenate([(s["labels"] > 2).astype(int) for s in test_single])
print(f"\nsingle-task test: {len(test_single)} sessions, {len(y_test_flat)} windows, "
      f"{y_test_flat.sum()} elevated")

ALIGNED = False
if FROZEN_PREDS.exists():
    frozen = np.load(FROZEN_PREDS)
    if len(frozen) != len(y_test_flat):
        raise SystemExit(f"frozen predictions have {len(frozen)} rows, rebuilt index has "
                         f"{len(y_test_flat)}; the window index does not match")
    auc_check = rank_auc(y_test_flat, frozen)
    ALIGNED = abs(auc_check - FROZEN_POOLED_AUC) < 1e-6
    print(f"alignment guard: rebuilt AUC {auc_check:.6f} vs frozen {FROZEN_POOLED_AUC:.6f} "
          f"-> {'PASS' if ALIGNED else 'FAIL'}")
    if not ALIGNED:
        raise SystemExit("window ordering does not reproduce the frozen endpoint; stopping "
                         "rather than reporting numbers that are not comparable")
else:
    print("alignment guard SKIPPED (no frozen predictions available)")

## Cell 3 — Split, standardize, pad

In [ ]:
# Cell 3: assemble padded tensors. Standardization is fitted on training windows only.
import numpy as np

SPLITS = {}
for name in ("train", "validation", "test"):
    SPLITS[name] = sorted(
        [s for s in SESSIONS if s["split"] == name and s["source"] in SINGLE_TASK_SOURCES],
        key=lambda s: (s["participant"], s["condition"]))
    n_w = sum(len(s["labels"]) for s in SPLITS[name])
    print(f"{name:11s} sessions={len(SPLITS[name]):4d} windows={n_w:5d} "
          f"participants={len({s['participant'] for s in SPLITS[name]})}")

train_flat = np.concatenate([s["features"].reshape(-1, len(FEATURE_COLUMNS))
                             for s in SPLITS["train"]]).astype(np.float64)
FEATURE_MEAN  = train_flat.mean(axis=0)
FEATURE_SCALE = train_flat.std(axis=0)
FEATURE_SCALE[FEATURE_SCALE == 0] = 1.0
del train_flat

PAD_LABEL = -1.0   # sentinel: the loss and every metric ignore steps with y < 0
LONGEST   = max(len(s["labels"]) for s in SESSIONS)
# pack() pads to the longest session in whatever set it is given; the model accepts a
# variable window axis, so nothing depends on a global maximum.
print(f"\nlongest session anywhere: {LONGEST} windows; pad sentinel {PAD_LABEL}")

def pack(sessions, add_clock=False, shuffle_rng=None, prefix_k=None):
    """Pad sessions into (B, W, 30, F) with labels (B, W) and a validity mask.

    add_clock    -- append normalized position-in-session as a 15th channel,
                    broadcast across the 30 timesteps of each window.
    shuffle_rng  -- permute window order within each session (the order control).
                    Windows and labels are permuted together, so the model sees
                    exactly the same (window, label) pairs in a random order.
    prefix_k     -- cut each session into consecutive chunks of at most k windows,
                    so recurrence can never reach further back than k.
    """
    prepared = []
    for s in sessions:
        feats = ((s["features"].astype(np.float64) - FEATURE_MEAN) / FEATURE_SCALE
                 ).astype(np.float32)
        labels, position, participant = s["labels"], s["position"], s["participant"]
        order = np.arange(len(labels))
        if shuffle_rng is not None:
            order = shuffle_rng.permutation(order)
            feats, labels, position = feats[order], labels[order], position[order]
        if add_clock:
            clock = np.repeat(position[:, None, None], WINDOW_LENGTH, axis=1)
            feats = np.concatenate([feats, clock], axis=-1)
        chunks = ([ (feats[i:i+prefix_k], labels[i:i+prefix_k], order[i:i+prefix_k])
                    for i in range(0, len(labels), prefix_k) ]
                  if prefix_k else [(feats, labels, order)])
        for cf, cl, co in chunks:
            prepared.append({"features": cf, "labels": cl,
                             "participant": participant, "order": co})

    width = max(len(p["labels"]) for p in prepared)
    n_feat = prepared[0]["features"].shape[-1]
    X = np.zeros((len(prepared), width, WINDOW_LENGTH, n_feat), dtype=np.float32)
    Y = np.full((len(prepared), width), PAD_LABEL, dtype=np.float32)
    for i, p in enumerate(prepared):
        k = len(p["labels"])
        X[i, :k] = p["features"]
        Y[i, :k] = p["labels"]
    participants = [p["participant"] for p in prepared]
    orders       = [p["order"] for p in prepared]
    return X, Y, participants, orders

X_demo, Y_demo, _, _ = pack(SPLITS["train"])
print("packed training tensor:", X_demo.shape, "| labels:", Y_demo.shape,
      "| valid steps:", int((Y_demo >= 0).sum()))
del X_demo, Y_demo

## Cell 4 — Models

The window encoder is the frozen architecture up to its `Dense(8)` layer, so the `window`
condition is the published model and the `session` condition is that same model with a causal
GRU added over the window axis. Any difference is attributable to the added recurrence and
nothing else.

In [ ]:
# Cell 4: model definitions.
import tensorflow as tf
from tensorflow.keras import layers

def window_encoder(n_features, name="encoder"):
    """Frozen CNN classifier body, truncated before the sigmoid.

    Mirrors build_cnn_classifier from analysis/composition_robustness.py exactly,
    including the leading Masking layer, so the window baseline reproduces the
    published architecture.
    """
    inp = layers.Input(shape=(WINDOW_LENGTH, n_features))
    x = layers.Masking(mask_value=0.0)(inp)
    x = layers.Conv1D(64, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.1)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(8, activation="relu")(x)
    return tf.keras.Model(inp, x, name=name)

def build_model(n_features, session_context, gru_units=16):
    """(B, W, 30, F) -> (B, W) elevation probabilities.

    session_context=False : each window scored independently (the published model,
                            applied in parallel across the window axis).
    session_context=True  : a causal GRU runs over the window embeddings, so the
                            score for window t may depend on windows 1..t.
    """
    inp = layers.Input(shape=(None, WINDOW_LENGTH, n_features))
    embedded = layers.TimeDistributed(window_encoder(n_features))(inp)   # (B, W, 8)
    if session_context:
        # Small on purpose: only ~127 training sessions constrain this layer.
        embedded = layers.GRU(gru_units, return_sequences=True,
                              dropout=0.1, name="session_gru")(embedded)
    out = layers.TimeDistributed(layers.Dense(1, activation="sigmoid"))(embedded)
    return tf.keras.Model(inp, layers.Reshape((-1,))(out))

def masked_weighted_bce(pos_weight, neg_weight):
    """BCE over valid steps only, with the frozen endpoint's balanced class weights.

    Padded steps carry y = PAD_LABEL (-1) and are removed by the mask, so padding
    contributes no gradient and no loss. Implemented directly rather than through
    sample_weight so that behaviour does not depend on the Keras version.
    """
    def loss(y_true, y_pred):
        valid  = tf.cast(tf.greater_equal(y_true, 0.0), tf.float32)
        target = tf.cast(tf.greater(y_true, 2.0), tf.float32)   # elevated = FMS > 2
        p = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        bce = -(target * tf.math.log(p) * pos_weight
                + (1 - target) * tf.math.log(1 - p) * neg_weight)
        return tf.reduce_sum(bce * valid) / tf.maximum(tf.reduce_sum(valid), 1.0)
    return loss

print("models defined")
print("window baseline parameters:",
      build_model(len(FEATURE_COLUMNS), session_context=False).count_params())
print("session model parameters:  ",
      build_model(len(FEATURE_COLUMNS), session_context=True).count_params())

## Cell 5 — Training and evaluation harness (participant-cluster bootstrap)

In [ ]:
# Cell 5: five-seed ensemble, per-window predictions retained, participant-cluster CI.
import numpy as np, tensorflow as tf, time

PREDICTIONS = {}   # condition -> {"scores", "labels", "participants"}; filled by run_condition

def participant_cluster_ci(y, scores, participants, replicates=BOOTSTRAP_REPLICATES, seed=42):
    """Resample whole participants, keeping every window from each sampled person."""
    rng = np.random.default_rng(seed)
    unique = np.array(sorted(set(participants)))
    index_of = {p: np.where(np.asarray(participants) == p)[0] for p in unique}
    draws = []
    for _ in range(replicates):
        picked = rng.choice(unique, size=len(unique), replace=True)
        idx = np.concatenate([index_of[p] for p in picked])
        value = rank_auc(y[idx], scores[idx])
        if not np.isnan(value):
            draws.append(value)
    if not draws:
        return {"lower": float("nan"), "upper": float("nan"), "replicates": 0}
    draws = np.array(draws)
    return {"lower": float(np.percentile(draws, 2.5)),
            "upper": float(np.percentile(draws, 97.5)),
            "replicates": int(len(draws))}

def run_condition(name, session_context, add_clock=False, shuffle=False,
                  prefix_k=None, epochs=150, patience=20, verbose=0):
    """Train the five-seed ensemble for one condition and score the held-out windows.

    The per-window scores are stored in PREDICTIONS so that conditions can later be
    compared on a shared participant resample. Every condition is evaluated on the
    same 768 windows in the same order -- shuffled and chunked variants are unpermuted
    before scoring -- which is what makes the paired comparison in Cell 6 valid.
    """
    started = time.time()
    rng_train = np.random.default_rng(0) if shuffle else None
    rng_val   = np.random.default_rng(1) if shuffle else None
    rng_test  = np.random.default_rng(2) if shuffle else None

    Xtr, Ytr, _,  _        = pack(SPLITS["train"],      add_clock, rng_train, prefix_k)
    Xva, Yva, _,  _        = pack(SPLITS["validation"], add_clock, rng_val,   prefix_k)
    Xte, Yte, te_part, te_order = pack(SPLITS["test"],  add_clock, rng_test,  prefix_k)

    valid_tr = Ytr >= 0
    elevated = (Ytr > 2) & valid_tr
    n_pos, n_neg = int(elevated.sum()), int((valid_tr & ~elevated).sum())
    total = n_pos + n_neg
    pos_w, neg_w = total / (2.0 * max(n_pos, 1)), total / (2.0 * max(n_neg, 1))

    n_features = Xtr.shape[-1]
    seed_scores = []
    for seed in ENSEMBLE_SEEDS:
        tf.keras.utils.set_random_seed(seed)
        model = build_model(n_features, session_context)
        model.compile(optimizer=tf.keras.optimizers.Adam(5e-4, clipnorm=1.0),
                      loss=masked_weighted_bce(pos_w, neg_w))
        model.fit(Xtr, Ytr, validation_data=(Xva, Yva), epochs=epochs, batch_size=8,
                  verbose=verbose,
                  callbacks=[tf.keras.callbacks.EarlyStopping(
                      monitor="val_loss", patience=patience, restore_best_weights=True)])
        seed_scores.append(model.predict(Xte, verbose=0))
        tf.keras.backend.clear_session()

    probabilities = np.mean(seed_scores, axis=0)            # (B, W)

    # Flatten back to the frozen window order: undo any shuffle, keep session order.
    flat_scores, flat_labels, flat_participants = [], [], []
    for i, participant in enumerate(te_part):
        valid = Yte[i] >= 0
        restore = np.argsort(te_order[i])
        flat_scores.append(probabilities[i][valid][restore])
        flat_labels.append((Yte[i][valid] > 2).astype(int)[restore])
        flat_participants.extend([participant] * int(valid.sum()))
    scores = np.concatenate(flat_scores)
    labels = np.concatenate(flat_labels)
    participants = np.asarray(flat_participants)

    # Every condition must land on the identical evaluation set, or the paired
    # comparison is meaningless. Check against the first condition run.
    if PREDICTIONS:
        reference = next(iter(PREDICTIONS.values()))
        if not np.array_equal(labels, reference["labels"]):
            raise SystemExit(f"{name}: label vector differs from earlier conditions; "
                             "the unpermutation is wrong and results are not comparable")
        if not np.array_equal(participants, reference["participants"]):
            raise SystemExit(f"{name}: participant vector differs from earlier conditions")

    PREDICTIONS[name] = {"scores": scores, "labels": labels, "participants": participants}

    auc = rank_auc(labels, scores)
    ci  = participant_cluster_ci(labels, scores, participants)
    result = {"condition": name, "protocol_version": PROTOCOL_VERSION,
              "session_context": session_context, "clock_feature": add_clock,
              "shuffled": shuffle, "prefix_k": prefix_k,
              "test_windows": int(len(labels)), "elevated": int(labels.sum()),
              "participants": int(len(set(participants.tolist()))),
              "pooled_auc": auc, "cluster_ci": ci,
              "minutes": round((time.time() - started) / 60, 1)}
    print(f"{name:22s} AUC={auc:.3f}  95% CI [{ci['lower']:.3f}, {ci['upper']:.3f}]  "
          f"n={len(labels)}  ({result['minutes']} min)")
    return result

def paired_cluster_difference(name_a, name_b, replicates=BOOTSTRAP_REPLICATES, seed=42):
    """95% interval on AUC(a) - AUC(b), resampling participants ONCE per replicate.

    Comparing two marginal intervals is a weak way to state a difference: both carry
    the same participant-sampling variance, so they overlap even when one model is
    consistently ahead. Scoring both models on the same resampled participants removes
    that shared variance and gives an interval on the difference itself.
    """
    a, b = PREDICTIONS[name_a], PREDICTIONS[name_b]
    y, participants = a["labels"], a["participants"]
    rng = np.random.default_rng(seed)
    unique = np.array(sorted(set(participants.tolist())))
    index_of = {p: np.where(participants == p)[0] for p in unique}
    diffs = []
    for _ in range(replicates):
        picked = rng.choice(unique, size=len(unique), replace=True)
        idx = np.concatenate([index_of[p] for p in picked])
        d_a, d_b = rank_auc(y[idx], a["scores"][idx]), rank_auc(y[idx], b["scores"][idx])
        if not (np.isnan(d_a) or np.isnan(d_b)):
            diffs.append(d_a - d_b)
    diffs = np.array(diffs)
    point = rank_auc(y, a["scores"]) - rank_auc(y, b["scores"])
    return {"comparison": f"{name_a} - {name_b}", "difference": float(point),
            "lower": float(np.percentile(diffs, 2.5)),
            "upper": float(np.percentile(diffs, 97.5)),
            "fraction_favouring_a": float((diffs > 0).mean()),
            "replicates": int(len(diffs))}

print("harness ready (per-window predictions retained; paired comparison available)")

## Cell 6 — Run all conditions

Order matters: the `window` baseline runs first and should land near **0.757**. If it does
not, stop — something in the environment differs from the frozen run and nothing downstream
is interpretable.

In [ ]:
# Cell 6: run every condition. ~35 min on a T4 at the timings observed so far.
import json, numpy as np

# Guard: Cell 5 defines both the harness and the PREDICTIONS store. Re-running an
# updated Cell 6 against a kernel that still holds the previous Cell 5 is the easiest
# mistake to make in Colab, so fail with an instruction rather than a NameError.
try:
    PREDICTIONS
    run_condition
except NameError:
    raise SystemExit(
        "Cell 5 has not been run in this kernel.\n"
        "Run Cell 5 (the harness) first, then re-run this cell.\n"
        "If you just applied a notebook update, re-run Cell 5 even if it ran before -- "
        "the kernel still holds the previous definition.")

RESULTS = []
PREDICTIONS.clear()

RESULTS.append(run_condition("window",           session_context=False))
baseline = RESULTS[-1]["pooled_auc"]
if not (0.70 <= baseline <= 0.81):
    print(f"\n*** WARNING: baseline {baseline:.3f} is far from the frozen 0.757. "
          f"Investigate before trusting anything below. ***\n")

RESULTS.append(run_condition("window_clock",     session_context=False, add_clock=True))
RESULTS.append(run_condition("session",          session_context=True))
RESULTS.append(run_condition("session_shuffled", session_context=True, shuffle=True))
for k in (2, 3, 5):
    RESULTS.append(run_condition(f"session_prefix_{k}", session_context=True, prefix_k=k))

# Per-window scores are the artifact every later comparison depends on, so they are
# written out rather than left in memory.
np.savez_compressed(
    OUTPUT_DIR / "session_scale_predictions.npz",
    labels=PREDICTIONS["window"]["labels"],
    participants=PREDICTIONS["window"]["participants"],
    **{f"scores__{name}": p["scores"] for name, p in PREDICTIONS.items()})
print("\nwrote", OUTPUT_DIR / "session_scale_predictions.npz")

out = OUTPUT_DIR / "session_scale_results.json"
out.write_text(json.dumps({"protocol_version": PROTOCOL_VERSION,
                           "frozen_pooled_auc": FROZEN_POOLED_AUC,
                           "alignment_guard_passed": bool(ALIGNED),
                           "ensemble_seeds": list(ENSEMBLE_SEEDS),
                           "results": RESULTS}, indent=2))
print("wrote", out)

## Cell 7 — Paired comparison against the window baseline

Two overlapping confidence intervals do not establish that two models differ, and two
non-overlapping ones are not required for them to. Both marginal intervals here are wide
mostly because there are only nine test participants, and that uncertainty is *shared* by
every condition — the same nine people are scored by all of them.

Resampling participants once per replicate and scoring every model on that same draw
removes the shared component and puts an interval on the difference itself. This is the
number to report.

In [ ]:
# Cell 7: paired participant-cluster intervals on the difference from the window baseline.
import numpy as np, json, pandas as pd

try:
    PREDICTIONS
    paired_cluster_difference
except NameError:
    raise SystemExit(
        "Cell 5 has not been run in this kernel. Run Cell 5, then Cell 6, then this cell.")
if "window" not in PREDICTIONS:
    raise SystemExit(
        "PREDICTIONS has no 'window' entry -- run Cell 6 first. If Cell 6 was run before "
        "the notebook was updated, run it again so per-window scores are retained.")

COMPARISONS = [paired_cluster_difference(name, "window")
               for name in PREDICTIONS if name != "window"]

frame = pd.DataFrame([{
    "comparison":  c["comparison"],
    "difference":  round(c["difference"], 3),
    "CI low":      round(c["lower"], 3),
    "CI high":     round(c["upper"], 3),
    "P(better)":   round(c["fraction_favouring_a"], 2),
} for c in COMPARISONS])
print(frame.to_string(index=False))

print("\ninterpretation:")
for c in COMPARISONS:
    crosses = c["lower"] <= 0 <= c["upper"]
    verdict = ("indistinguishable from the window baseline" if crosses
               else ("BETTER than the window baseline" if c["difference"] > 0
                     else "WORSE than the window baseline"))
    print(f"  {c['comparison']:34s} {c['difference']:+.3f} "
          f"[{c['lower']:+.3f}, {c['upper']:+.3f}]  {verdict}")

path = OUTPUT_DIR / "session_scale_paired_comparisons.json"
path.write_text(json.dumps(COMPARISONS, indent=2))
print("\nwrote", path)

print("\nNote: a paired interval that contains zero is a statement that this experiment")
print("cannot separate the two models on nine participants. It is not evidence that they")
print("are identical, and it is not weakened by the marginal intervals overlapping.")

## Cell 8 — Read the result

In [ ]:
# Cell 8: summary table and the three questions this notebook exists to answer.
import pandas as pd, numpy as np

try:
    RESULTS
except NameError:
    raise SystemExit("run Cell 6 first -- RESULTS is undefined")

table = pd.DataFrame([{
    "condition": r["condition"],
    "AUC": round(r["pooled_auc"], 3),
    "CI low": round(r["cluster_ci"]["lower"], 3),
    "CI high": round(r["cluster_ci"]["upper"], 3),
    "windows": r["test_windows"],
    "min": r["minutes"],
} for r in RESULTS])
print(table.to_string(index=False))

get = lambda name: next((r["pooled_auc"] for r in RESULTS if r["condition"] == name), np.nan)
window, clock, session, shuffled = (get("window"), get("window_clock"),
                                    get("session"), get("session_shuffled"))

print("\n" + "=" * 72)
print(f"frozen endpoint                        0.757")
print(f"clock alone (measured previously)      0.564")
print(f"window baseline, retrained here        {window:.3f}")
print(f"session context                        {session:.3f}   delta {session - window:+.3f}")
print("=" * 72)

print("\n1. Did session context help?")
print(f"   {session - window:+.3f} AUC. Compare against the composition spread the article "
      f"already reports (SD 0.083):\n   a delta well inside that spread is not a finding.")

print("\n2. Was it temporal order, or just pooling?")
if not np.isnan(shuffled):
    print(f"   shuffled context {shuffled:.3f} vs ordered {session:.3f} "
          f"(difference {session - shuffled:+.3f}).")
    print("   If shuffling costs little, the gain came from seeing more data per example,")
    print("   NOT from accumulation. That is a publishable negative result -- report it.")

print("\n3. Was it just a clock?")
if not np.isnan(clock):
    print(f"   window+clock {clock:.3f} vs session {session:.3f}.")
    print("   If the clock alone recovers most of the gain, the GRU learned elapsed time,")
    print("   not physiological accumulation. Say so explicitly in the text.")

prefix = [(r["prefix_k"], r["pooled_auc"]) for r in RESULTS if r["prefix_k"]]
if prefix:
    print("\n4. How far back does the signal reach?")
    for k, auc in sorted(prefix):
        print(f"   context <= {k:2d} windows ({k * 15 + 15:3d} s): AUC {auc:.3f}")
    print(f"   full session:                    AUC {session:.3f}")
    print("   A curve that plateaus early bounds the useful temporal scale -- which is a")
    print("   sharper claim than the article can currently make in either direction.")

print("\nREMINDER: this is one participant assignment. By the article's own argument, a")
print("gain here needs the 20-composition treatment before it can be a headline number.")